In [1]:
import os
base_path = os.path.dirname(os.path.abspath(""))
base_path = base_path + "/headers/"
print(base_path)
header_path = base_path + 'header_data_treatment.ipynb'
%run $header_path

C:\Users\raphael.wargon\Documents\GitHub\sorting_univ\script/headers/
initalizing spark nlp session w gpu


In [2]:
works_au_af = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'works_au_af.parquet')
               .filter( func.col('publication_year').between(1950,2020) )
              )

In [3]:
df_authors = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'authors.parquet')
              )

In [4]:
journals_ranking = (spark.read.format('parquet').load('file:\\' + main_path_openalex + 'journals_ranking_year.parquet')
                    .withColumn('source_id', func.regexp_replace(func.col('source_id'), 'https://openalex.org/', ''))
                   )
journals_ranking.show()

+-----------+----+-----------+-------------------+------------------+
|  source_id|year|source_type|    rank_source_pct|   abs_rank_source|
+-----------+----+-----------+-------------------+------------------+
| S100014387|2021|    journal|                0.0|               1.0|
| S100014455|2010|    journal| 0.8969174714671388|1275.9381443298969|
|S1000263748|2002|    journal| 0.8676467635588629| 872.2857142857142|
|S1000263748|2019|    journal| 0.8747890323638025|3337.1707317073174|
| S100028386|2005|    journal| 0.8903641881638846|             284.0|
| S100045985|1964|    journal|  0.909952606635071|              96.0|
| S100060201|2011|    journal| 0.8432467959250739|            2382.0|
| S100062655|2015|    journal| 0.9088593802466257|            1650.0|
| S100062975|2020|    journal| 0.9048618716892145| 2239.538461538462|
|  S10007838|1972|    journal|0.15706909290143697| 839.8000000000001|
|S1000839972|1979|    journal|  0.679080824088748|             387.0|
|S1000839972|2018|  

In [5]:
#df_au_scanR = spark.read.format('parquet').load('file:\\' + scanR_path + '\\authors.parquet')

In [6]:
papers_textual_metrics = spark.read.csv('file:///' + databases_path + '/papers_textual_metrics.csv', header = True) 
papers_textual_metrics = papers_textual_metrics.withColumnRenamed('paperID', 'work_id').drop(*['n_words','n_phrases', 'has_abstract'])

In [7]:
papers_textual_metrics.show()

+-----------+--------+--------------+----------+----------------+-------------+-------------------+---------------+---------------------+--------------------+
|    work_id|new_word|new_word_reuse|new_phrase|new_phrase_reuse|new_word_comb|new_word_comb_reuse|new_phrase_comb|new_phrase_comb_reuse|   semantic_distance|
+-----------+--------+--------------+----------+----------------+-------------+-------------------+---------------+---------------------+--------------------+
|W2026081217|       0|             0|         0|               0|            0|                  0|              4|                   91| 0.18517166376113892|
|W2330301979|       0|             0|         0|               0|            0|                  0|              0|                    0|  0.1968483328819275|
|W2327188017|       0|             0|         0|               0|           56|                375|             16|                   51| 0.14563798904418945|
|W2107594894|       0|             0|         

# Build set of French and UK authors

In [12]:
inst = spark.read.parquet('file:\\' + int_save_path + 'inst_fr.parquet')

In [13]:
inst_UK = spark.read.parquet('file:\\' + int_save_path + 'inst_UK.parquet')

In [10]:
inst.filter(func.col('inst_id').contains('I4210115196')).show(truncate  = False)

+-----------+----------------------------------------------------+--------+---------+---------------------------+-------------------------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+-------------------------------------+----+----+-------------------+----------+-------+---------+---------------+-----------+--------------+
|inst_id    |name                                                |type    |city     |homepage_url               |topic_share                    |main_topic      |parent                                                                                                                                                                                      |uni_pub|parent_id                            |cnrs|idex|total_idex_dotation|type_fr   |secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|


In [10]:
inst.filter(func.col('inst_id')=="I118618916").show()

+----------+--------------------+---------+--------------------+--------------------+--------------------+---------------+--------------------+-------+----+----+-------------------+-------------------+-------+---------+---------------+-----------+--------------+
|   inst_id|                name|     type|                city|        homepage_url|         topic_share|     main_topic|              parent|uni_pub|cnrs|idex|total_idex_dotation|            type_fr|secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|
+----------+--------------------+---------+--------------------+--------------------+--------------------+---------------+--------------------+-------+----+----+-------------------+-------------------+-------+---------+---------------+-----------+--------------+
|I118618916|Université Pierre...|education|Saint-Martin-d'Hères|http://www.upmf-g...|[{Social Sciences...|Social Sciences|[{I899635006, Uni...|      1|   0|NULL|               NULL|Grand établissement| public|  

In [11]:
set_french_UK_authors = (works_au_af
                      .filter(func.col('publication_year').between(1997,2020))
                     # .withColumn('inst_id', func.when(func.col('inst_id')=='I118618916',  "I899635006")
                     #                            .when(func.col('inst_id')=="I36085230",  "I899635006")
                     #                            .when(func.col('inst_id')=="I177483745",  "I899635006")
                     #                            .when(func.col('inst_id')=="I4210103002", "I198244214")
                     #                            .when(func.col('inst_id')=="I4210143836", "I198244214")
                     #                            .when(func.col('inst_id')=="I184646667",  "I39804081")
                     #                            .when(func.col('inst_id')=="I102197404",  "I277688954")
                     #                            .when(func.col('inst_id')=='I7171862',  "I2279609970")
                     #                            .when(func.col('inst_id')=="I59807433",  "I2279609970")
                     #                            .when(func.col('inst_id')=="I2800379142", "I4210154111") 
                     #                            .when(func.col('inst_id')=="I3123023596","I56067802")
                     #                            .when(func.col('inst_id')=="I4210095130","I208215962")
                     #                            .otherwise(func.col('inst_id')))
                      .join(
                          inst.select('inst_id', func.lit(1).alias('in_inst_dataset'), func.lit('FR').alias('inst_dataset'))
                          .unionAll(inst_UK.select('inst_id', func.lit(1).alias('in_inst_dataset'), func.lit('UK').alias('inst_dataset')))
                          , on =['inst_id'], how = 'left')
                      .fillna(0, subset = ['in_inst_dataset'])
                      #.filter( (func.col('country')=='FR') | (func.col('in_inst_dataset')==1))
                      .groupBy('author_id')
                      .agg(func.max('publication_year').alias('last_year'),
                           func.min('publication_year').alias('entry_year'),
                           func.countDistinct('work_id').alias('n_pub_in_ds'),
                           func.countDistinct('publication_year').alias('n_y_in_ds')
                          )
                      .filter( ~(func.col('last_year')==func.col('entry_year'))
                               & (func.col('n_y_in_ds')>2)
                               & (func.col('n_pub_in_ds')>2)
                             )
                     )
#set_french_authors.cache()
#set_french_authors.count()

In [13]:
#(set_french_authors.filter(func.col('author_id')=="A5084984675")
#                      .join(works_au_af, on = ['author_id'], how = 'left')).show()

In [14]:
#set_french_authors.show()

In [15]:
set_french_UK_authors.write.mode('overwrite').parquet('file:\\' + int_save_path + 'set_french_UK_authors.parquet')

# Load set of French authors

In [8]:
set_french_UK_authors = spark.read.parquet('file:\\' + int_save_path + 'set_french_UK_authors.parquet')
set_french_UK_authors.count()

15452551

In [9]:
set_french_UK_authors.show()

+-----------+---------+----------+-----------+---------+
|  author_id|last_year|entry_year|n_pub_in_ds|n_y_in_ds|
+-----------+---------+----------+-----------+---------+
|A5000041874|     2020|      1999|         75|       19|
|A5000054118|     2016|      1999|          5|        4|
|A5000066187|     2020|      1997|          7|        6|
|A5000073587|     2020|      1997|         70|       22|
|A5000102890|     2020|      2008|         43|       12|
|A5000164563|     2014|      2010|          3|        3|
|A5000182388|     2014|      1999|         12|        8|
|A5000201649|     2020|      2007|         37|       14|
|A5000202654|     2020|      2018|          4|        3|
|A5000213748|     2020|      1997|        551|       23|
|A5000216468|     2020|      2011|         20|        6|
|A5000241720|     2020|      2015|        344|        6|
|A5000264900|     2020|      2002|         48|       14|
|A5000288583|     2020|      1999|        185|       17|
|A5000331660|     2020|      19

In [10]:
works_au_af.show()


+-----------+-----------+-----------+-----------+----------------+---------+------------+---------------+-------+---------+-------------------------------------+--------+----------------+--------------------+-----------------------+-------+------+
|    work_id|  author_id|    inst_id|  source_id|publication_year|citations|        type|author_position|country|inst_type|                                title|language|         version|       primary_topic|raw_affiliation_strings|funders|awards|
+-----------+-----------+-----------+-----------+----------------+---------+------------+---------------+-------+---------+-------------------------------------+--------+----------------+--------------------+-----------------------+-------+------+
|W1000000251|A5111984641|I1292966370|       NULL|            1990|        0|book-chapter|           NULL|     GB|    other|                 Antitumor Effects...|      en|publishedVersion|{https://openalex...|   [Drug Targeting L...|   NULL|  NULL|
|W100000

In [11]:
works_au_af_subset = (set_french_UK_authors
                      .join(works_au_af, on = ['author_id'], how = 'left')
                     )
#works_au_af_subset.cache()#
#works_au_af_subset.show()

In [14]:
works_au_af_subset = (works_au_af_subset
                      .withColumnRenamed('type', 'work_type')
                 # .withColumn('inst_id', func.expr(expr_inst_id))
                 # .join(inst.drop('country').distinct(), on = ['inst_id'])
                  .join(inst.distinct()
                        .unionAll(inst_UK.select(*inst_UK.columns,
                                                 *[func.lit(None).alias(col) for col in inst.columns if col not in inst_UK.columns])
                                  .select(inst.columns) .distinct())
                        .withColumn('in_inst', func.lit(1)), on = ['inst_id'], how = 'left')
                    .withColumn('inst_id', func.when(func.col('inst_id').isNull(), 'abroad')
                                .otherwise(func.col('inst_id')))
                     )
#works_au_af_subset.cache()
#works_au_af_subset.show()

In [15]:
works_au_af_subset.distinct().write.mode('overwrite').parquet('file:\\' + int_save_path + 'works_au_af_subset.parquet') 

In [16]:
works_au_af_subset = spark.read.parquet('file:\\' + int_save_path + 'works_au_af_subset.parquet')

In [17]:
works_au_af_subset.count()

484623562

In [18]:
works_au_af_subset.show()

+-------+-----------+---------+----------+-----------+---------+-----------+-----------+----------------+---------+---------+---------------+-------+---------+--------------------+--------+-------+--------------------+-----------------------+-------+------+----+----+----+------------+-----------+----------+------+-------+---------+----+----+-------------------+-------+-------+---------+---------------+-----------+--------------+-------+
|inst_id|  author_id|last_year|entry_year|n_pub_in_ds|n_y_in_ds|    work_id|  source_id|publication_year|citations|work_type|author_position|country|inst_type|               title|language|version|       primary_topic|raw_affiliation_strings|funders|awards|name|type|city|homepage_url|topic_share|main_topic|parent|uni_pub|parent_id|cnrs|idex|total_idex_dotation|type_fr|secteur|acces_rce|date_first_idex|fusion_date|merged_inst_id|in_inst|
+-------+-----------+---------+----------+-----------+---------+-----------+-----------+----------------+---------+---

# Build author-level characteristics


In [19]:
au_level_char = (works_au_af_subset
                                       .join(inst.select('inst_id', func.lit(1).alias('in_inst_dataset')), on =['inst_id'], how = 'left')
                      .fillna(0, subset = ['in_inst_dataset'])
              .groupBy('author_id','publication_year').agg(func.max((func.col('in_inst_dataset')==1).cast('int')).alias('in_FR'),
                                                          func.max((func.col('in_inst_dataset')==0).cast('int')).alias('abroad'))
              .groupBy('author_id').agg(func.min('publication_year').alias('entry_year'),
                                        func.max('publication_year').alias('last_year'),
                                        func.sum('in_FR').alias('all_y_in_sample'), 
                                        func.sum('abroad').alias('years_abroad'))
                )
au_level_char.cache()
au_level_char.show()

+-----------+----------+---------+---------------+------------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|
+-----------+----------+---------+---------------+------------+
|A5000015876|      2008|     2020|              0|           9|
|A5000020279|      2004|     2017|              0|           7|
|A5000020320|      1993|     2020|              0|          28|
|A5000021849|      1995|     2015|              0|           8|
|A5000026392|      2010|     2014|              0|           3|
|A5000031401|      1990|     2007|              0|           8|
|A5000032138|      1994|     2020|              0|          18|
|A5000033050|      2003|     2018|              0|           7|
|A5000042988|      2011|     2020|              0|           4|
|A5000057287|      1975|     2020|              0|          20|
|A5000060417|      1994|     2020|              0|          21|
|A5000062282|      2004|     2018|              0|           8|
|A5000076855|      1993|     2007|      

In [20]:
au_level_char = (au_level_char
                 .join(works_au_af_subset.groupBy('author_id', func.col('publication_year').alias('entry_year'))
                       .agg(func.concat_ws(',', func.array_sort(func.array_compact(func.collect_set('country'))))
                            .alias('entry_country')),
                       on = ['author_id', 'entry_year'], how = 'left')
                )

In [21]:
au_level_char.filter(func.col('author_id')=="A5084984675").show()

+-----------+----------+---------+---------------+------------+-------------+
|  author_id|entry_year|last_year|all_y_in_sample|years_abroad|entry_country|
+-----------+----------+---------+---------------+------------+-------------+
|A5084984675|      1985|     2020|             28|          35|        FR,US|
+-----------+----------+---------+---------------+------------+-------------+



In [22]:
gendersort_path =  os.path.dirname(os.path.abspath("")) + "\\additional_material\\Wiki_Gendersort.py"
%run $gendersort_path

In [23]:
udf_wiki_gendersort = func.udf(wiki_gendersort().assign,StringType())
au_fields = (set_french_UK_authors
             .join(df_authors
                  .select(func.regexp_replace(func.col('id'), 'https://openalex.org/', '').alias('author_id'), func.col('display_name').alias('name'),
                    func.explode(func.col('topic_share')).alias('topic_share'))
                   , on = ['author_id'], how = 'inner')
             .withColumn('domain', func.col('topic_share.domain.id')) 
             .withColumn('max_topic', (func.col('topic_share.value') ==
                                                   func.max(func.col('topic_share.value'))
                                          .over(Window.partitionBy('author_id'))).cast('int'))
             .withColumn('topic', func.array_compact(func.collect_set(func.when(func.col('max_topic')==1, func.col('topic_share.id')))
                                                        .over(Window.partitionBy('author_id')) ) )
             .withColumn('subfield_value', func.sum(func.col('topic_share.value'))
                         .over(Window.partitionBy('author_id', 'topic_share.subfield.id')))
             .withColumn('max_subfield', func.max(func.col('subfield_value')).over(Window.partitionBy("author_id")))
             .withColumn('subfield', func.array_compact(func.collect_set(func.when(func.col('subfield_value')>=0.9*func.col('max_subfield'), func.col('topic_share.subfield.id')))
                                                        .over(Window.partitionBy('author_id')) ) )
            
             
             .groupBy('author_id','name', 'domain', 'topic', 'subfield',
                      func.col('topic_share.field.id').alias('field')
                     )
             .agg(func.sum("topic_share.value").alias('value'))
             .withColumn('max_value', func.max(func.col('value')).over(Window.partitionBy('author_id')))
             .filter(func.col('value') >= 0.75*func.col('max_value'))
             .groupBy('author_id', 'name','topic', 'subfield')
             .agg(*[ func.array_sort(func.collect_set(func.col(col))).alias(col)
                  for col in ['domain','field']])
             .withColumn('domain',    func.concat_ws(',',func.array_sort(func.transform(func.col('domain'), lambda x: func.regexp_replace(x, 'https://openalex.org/domains/', '')))))
             .withColumn('field',     func.concat_ws(',',func.array_sort(func.transform(func.col("field"), lambda x: func.regexp_replace(x, 'https://openalex.org/fields/', '')))))
             .withColumn('subfield',  func.concat_ws(',',func.array_sort(func.transform(func.col("subfield"), lambda x: func.regexp_replace(x, 'https://openalex.org/subfields/', '')))))
             .withColumn('topic',     func.concat_ws(',',func.array_sort(func.transform(func.col("topic"), lambda x: func.regexp_replace(x, 'https://openalex.org/', '')))))

            .withColumn('author_first_name', func.element_at(func.split(func.col('name'), ' '), 1))
            .withColumn('topic', func.when(func.length(func.col('topic'))<1, func.lit(None)).otherwise(func.col('topic')))
            .withColumn('topic', func.when(func.length(func.col('subfield'))<1, func.lit(None)).otherwise(func.col('subfield')))

            .select('author_id', func.col('name').alias('author_name'),
                    *[func.concat_ws(',', (func.coalesce(func.col(col), func.col(col +'')))).alias(col) for col in ['domain','field','subfield','topic'] ],
                   func.coalesce(func.col('author_first_name'), func.col('author_first_name')).alias('author_first_name'))

            .withColumn('gender', udf_wiki_gendersort(func.col('author_first_name')))
           )
au_fields

DataFrame[author_id: string, author_name: string, domain: string, field: string, subfield: string, topic: string, author_first_name: string, gender: string]

In [24]:
#au_level_char.count()

In [25]:
au_level_char = (au_level_char
                 .filter((func.col('entry_year')<2020) & (func.col('entry_year')>1954))
                 .join(au_fields, on = ['author_id'], how = 'left')
                 .distinct()
                )

In [26]:
au_level_char

DataFrame[author_id: string, entry_year: int, last_year: int, all_y_in_sample: bigint, years_abroad: bigint, entry_country: string, author_name: string, domain: string, field: string, subfield: string, topic: string, author_first_name: string, gender: string]

In [27]:
#au_level_char.show()

In [28]:
au_level_char.write.mode('overwrite').parquet('file:\\' + int_save_path + 'au_level_char.parquet')

In [29]:
spark.catalog.clearCache()
spark.stop()